# Lembar Kerja Mahasiswa (LKM)
## Praktikum Pertemuan 03 — Siklus Hidup Machine Learning dengan Scikit-Learn

Lembar kerja ini merupakan **tugas mandiri** dari modul `lec03` (Pertemuan 03). Kerjakan seluruh kegiatan secara berurutan, isi setiap tabel dan kotak jawaban, lalu simpan hasilnya sebagai PDF.

---

### Identitas Mahasiswa

*Klik dua kali sel ini untuk mengedit, isi titik-titik di bawah, lalu tekan `Shift + Enter`.*

| | |
|---|---|
| **Nama** | ............................................................ |
| **NIM** | ............................................................ |
| **Kelas / Rombel** | ............................................................ |
| **Nama Dosen** | ............................................................ |
| **Tanggal Praktikum** | ............................................................ |

---

### Capaian yang Diukur

Setelah menyelesaikan lembar kerja ini, Anda diharapkan mampu:

1. Menjelaskan pengaruh pembagian data acak terhadap hasil evaluasi model.
2. Membuktikan secara empiris manfaat standarisasi fitur.
3. Membandingkan dua keluarga model beserta konsekuensi biayanya.
4. Menafsirkan confusion matrix untuk menemukan kelemahan model.
5. Memilih metrik evaluasi yang sesuai selain akurasi.


---
## Petunjuk Pengerjaan

**Baca bagian ini sebelum mulai.**

1. Jalankan sel kode **berurutan dari atas ke bawah**. Banyak sel bergantung pada hasil sel sebelumnya.
2. Sel kode yang berisi tanda `____` atau komentar `# TODO` **harus Anda lengkapi sendiri**. Sel akan error bila dijalankan sebelum dilengkapi — itu normal.
3. Setiap kegiatan memiliki **tabel hasil** dan **kotak jawaban** berupa sel markdown. Klik dua kali sel tersebut untuk mengedit, lalu tekan `Shift + Enter` agar tampilannya rapi kembali.
4. Jawaban analisis dinilai dari **penalaran**, bukan panjangnya. Dua sampai empat kalimat yang tepat lebih bernilai daripada satu paragraf yang mengambang.
5. Angka pada tabel hasil **harus berasal dari eksekusi di komputer Anda sendiri**, bukan disalin dari teman.

> **Penting.** Nilai `random_state` pada lembar kerja ini diturunkan dari NIM Anda, sehingga hasil setiap mahasiswa akan sedikit berbeda. Jangan menyalin angka orang lain.

### Perkiraan waktu

| Bagian | Perkiraan |
|---|---|
| Persiapan (unduh dan muat data) | 5–10 menit |
| Kegiatan 1 sampai 5 | 60–75 menit |
| Kesimpulan dan ekspor PDF | 10 menit |

### Cara menyimpan sebagai PDF

Setelah semua kegiatan selesai dan **seluruh sel sudah dijalankan** (agar keluarannya ikut tercetak):

* **JupyterLab:** menu `File` → `Save and Export Notebook As...` → `HTML`, lalu buka berkas HTML di browser dan cetak (`Ctrl + P`) dengan tujuan **Save as PDF**.
* **Jupyter Notebook klasik:** `File` → `Download as` → `HTML`, lalu cetak ke PDF seperti di atas.
* **Google Colab:** `File` → `Print` → tujuan **Save as PDF**.
* **Baris perintah:** jalankan `jupyter nbconvert --to html LKM_Pertemuan03.ipynb` lalu cetak berkas HTML-nya ke PDF.

Jalur lewat HTML dianjurkan karena ekspor PDF langsung membutuhkan LaTeX yang sering belum terpasang.


---
## Persiapan

Bagian ini **sudah lengkap** — Anda hanya perlu menjalankannya. Bagian ini memuat data, menyiapkan fungsi bantu, dan membangun model dasar yang akan dipakai pada Kegiatan 3 sampai 5.


### P.1 Memuat pustaka

In [ ]:
import time
import numpy as np
import pandas as pd
import plotly.express as px

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

print("Semua pustaka berhasil dimuat.")

### P.2 Mengisi NIM Anda

Ganti angka di bawah dengan **tiga digit terakhir NIM** Anda. Angka ini dipakai sebagai `random_state` pribadi sehingga hasil Anda berbeda dari mahasiswa lain.

In [ ]:
# TODO: ganti dengan tiga digit terakhir NIM Anda, misalnya 137
NIM_3_DIGIT = ____

SEED = int(NIM_3_DIGIT)
print("random_state pribadi Anda (SEED) =", SEED)

### P.3 Memuat dataset Fashion-MNIST

Unduhan berukuran sekitar 30 MB dan hanya terjadi satu kali.

> Agar praktikum tidak terlalu lama, kita memakai **subhimpunan 12.000 gambar** dari 60.000 gambar yang tersedia. Semua kesimpulan yang akan kita tarik tetap berlaku, hanya angkanya sedikit berbeda dari dataset penuh.

In [ ]:
import torchvision

data = torchvision.datasets.FashionMNIST(root='data', train=True, download=True)

images_all = data.data.numpy().astype(float)
targets = data.targets.numpy()
class_dict = {i: nama for i, nama in enumerate(data.classes)}
labels_all = np.array([class_dict[t] for t in targets])

# Ambil subhimpunan acak 12.000 gambar agar praktikum berjalan cepat
N_SUBSET = 12000
rng = np.random.default_rng(SEED)
idx_subset = rng.choice(len(images_all), size=N_SUBSET, replace=False)

images = images_all[idx_subset]
labels = labels_all[idx_subset]

print("Jumlah gambar yang dipakai :", len(images))
print("Ukuran satu gambar         :", images[0].shape)
print("Daftar kelas               :", list(class_dict.values()))

### P.4 Fungsi bantu

Dua fungsi berikut dipakai berulang kali. Anda tidak perlu memahami detail `show_images`.

In [ ]:
def flatten(images):
    """Mengubah tiap gambar 28x28 menjadi satu vektor berdimensi 784."""
    return images.reshape(images.shape[0], -1)


def show_images(images, max_images=40, ncols=5, labels=None):
    """Menampilkan sebagian gambar dari dataset dalam bentuk kisi."""
    n = min(images.shape[0], max_images)
    px_height = 220
    fig = px.imshow(images[:n, :, :], color_continuous_scale='gray_r',
                    facet_col=0, facet_col_wrap=ncols,
                    height=px_height * int(np.ceil(n / ncols)))
    fig.update_layout(coloraxis_showscale=False)
    if labels is not None:
        fig.for_each_annotation(lambda a: a.update(text=labels[int(a.text.split("=")[-1])]))
    return fig


print("Fungsi bantu siap dipakai.")

### P.5 Pembagian data dan model dasar

Di sini kita membuat pembagian **train / validation / test** memakai `SEED` Anda, melakukan standarisasi, lalu melatih model regresi logistik dasar.

> Sel pelatihan di bawah membutuhkan waktu sekitar **1–3 menit**. Tunggu sampai tanda `[*]` di samping sel berubah menjadi angka.

In [ ]:
# Pembagian data: 64% latih, 16% validasi, 20% uji
images_tr, images_te, labels_tr, labels_te = train_test_split(
    images, labels, test_size=0.2, random_state=SEED)

images_tr, images_val, labels_tr, labels_val = train_test_split(
    images_tr, labels_tr, test_size=0.2, random_state=SEED)

print("Data latih   :", images_tr.shape)
print("Data validasi:", images_val.shape)
print("Data uji     :", images_te.shape)

In [ ]:
# Standarisasi: fit HANYA pada data latih
image_scaler = StandardScaler()
image_scaler.fit(flatten(images_tr))


def featurizer(images):
    return image_scaler.transform(flatten(images))


X_tr = featurizer(images_tr)
X_val = featurizer(images_val)
X_te = featurizer(images_te)

print("Dimensi fitur data latih:", X_tr.shape)

In [ ]:
# Melatih model dasar (regresi logistik) — butuh 1-3 menit
t0 = time.time()
lr_model = LogisticRegression(tol=1e-3, max_iter=200, random_state=SEED)
lr_model.fit(X=X_tr, y=labels_tr)
waktu_lr = time.time() - t0

print("Selesai dilatih dalam {:.1f} detik.".format(waktu_lr))
print("Akurasi validasi:", accuracy_score(labels_val, lr_model.predict(X_val)))

---
# Kegiatan 1 — Pengaruh Pembagian Data Acak

**Rujukan:** Latihan mandiri nomor 1 pada `lec03`.

**Pertanyaan yang ingin dijawab:** apakah akurasi model berubah bila pembagian data acaknya diganti? Sebesar apa? Apa artinya bagi cara kita melaporkan hasil?

### Langkah kerja

1. Lengkapi fungsi `eksperimen()` di bawah.
2. Jalankan fungsi itu untuk tiga nilai `random_state` yang berbeda.
3. Isi tabel hasil.
4. Jawab pertanyaan analisis.


### 1.1 Lengkapi fungsi eksperimen

Fungsi ini melakukan satu putaran penuh: bagi data → standarisasi → latih → ukur akurasi. Lengkapi bagian bertanda `____`.

> **Ingat aturan pentingnya.** `fit` pada `StandardScaler` hanya boleh dilakukan pada **data latih**. Perhatikan baik-baik variabel mana yang Anda masukkan.

In [ ]:
def eksperimen(seed, gunakan_standarisasi=True):
    """Satu putaran penuh: bagi data, standarisasi, latih, ukur akurasi.

    Mengembalikan dict berisi akurasi latih, akurasi uji, dan lama pelatihan.
    """
    # TODO (a): bagi data menjadi latih dan uji dengan proporsi uji 20%
    #           dan random_state sesuai argumen `seed`
    im_tr, im_te, lb_tr, lb_te = train_test_split(
        images, labels, test_size=____, random_state=____)

    if gunakan_standarisasi:
        scaler = StandardScaler()
        # TODO (b): pelajari statistik dari data LATIH saja
        scaler.fit(flatten(____))
        Xtr = scaler.transform(flatten(im_tr))
        Xte = scaler.transform(flatten(im_te))
    else:
        # Tanpa standarisasi: pakai nilai piksel mentah
        Xtr = flatten(im_tr)
        Xte = flatten(im_te)

    t0 = time.time()
    model = LogisticRegression(tol=1e-3, max_iter=200, random_state=42)
    model.fit(X=Xtr, y=lb_tr)
    lama = time.time() - t0

    # TODO (c): hitung akurasi pada data latih dan data uji
    #           gunakan accuracy_score(label_sebenarnya, prediksi)
    acc_tr = accuracy_score(lb_tr, model.____(Xtr))
    acc_te = accuracy_score(____, model.predict(Xte))

    return {"akurasi_latih": acc_tr, "akurasi_uji": acc_te, "lama_detik": lama}


print("Fungsi eksperimen siap. Jalankan sel berikutnya untuk mengujinya.")

### 1.2 Jalankan untuk tiga nilai random_state

Sel di bawah melatih tiga model. Waktunya sekitar **3–6 menit**.

In [ ]:
daftar_seed = [SEED, 42, 2025]
hasil_k1 = []

for s in daftar_seed:
    print("Menjalankan eksperimen dengan random_state =", s, "...")
    r = eksperimen(s, gunakan_standarisasi=True)
    r["random_state"] = s
    hasil_k1.append(r)
    print("   akurasi uji = {:.4f}".format(r["akurasi_uji"]))

tabel_k1 = pd.DataFrame(hasil_k1)[["random_state", "akurasi_latih", "akurasi_uji", "lama_detik"]]
tabel_k1

In [ ]:
# Ringkasan sebaran akurasi uji
akurasi_uji = tabel_k1["akurasi_uji"]

print("Akurasi uji terendah  : {:.4f}".format(akurasi_uji.min()))
print("Akurasi uji tertinggi : {:.4f}".format(akurasi_uji.max()))
print("Selisih (rentang)     : {:.4f}".format(akurasi_uji.max() - akurasi_uji.min()))
print("Rata-rata             : {:.4f}".format(akurasi_uji.mean()))
print("Simpangan baku        : {:.4f}".format(akurasi_uji.std()))

### 1.3 Tabel Hasil Pengamatan

*Salin angka dari keluaran sel di atas ke dalam tabel berikut. Klik dua kali untuk mengedit.*

| random_state | Akurasi latih | Akurasi uji | Lama pelatihan (detik) |
|---|---|---|---|
| (SEED Anda) = ...... | ...... | ...... | ...... |
| 42 | ...... | ...... | ...... |
| 2025 | ...... | ...... | ...... |

| Ringkasan | Nilai |
|---|---|
| Akurasi uji terendah | ...... |
| Akurasi uji tertinggi | ...... |
| Selisih (rentang) | ...... |
| Simpangan baku | ...... |


### 1.4 Pertanyaan Analisis

**A1.** Apakah akurasi uji berubah ketika `random_state` diganti? Sebutkan besar selisih antara nilai tertinggi dan terendah pada percobaan Anda.

> *Jawaban Anda:*
>
> ......

**A2.** Bayangkan seorang mahasiswa melaporkan "model saya mencapai akurasi 85,3%" tanpa menyebutkan `random_state`. Mengapa laporan seperti itu bermasalah?

> *Jawaban Anda:*
>
> ......

**A3.** Apa cara pelaporan yang lebih jujur untuk hasil seperti pada tabel 1.3? Sebutkan minimal satu cara.

> *Jawaban Anda:*
>
> ......


---
# Kegiatan 2 — Manfaat Standarisasi Fitur

**Rujukan:** Latihan mandiri nomor 2 pada `lec03`.

**Pertanyaan yang ingin dijawab:** apa yang terjadi bila model dilatih langsung dari nilai piksel mentah (0–255) tanpa standarisasi?

### Langkah kerja

Kita memakai fungsi `eksperimen()` yang sama, tetapi kali ini dengan `gunakan_standarisasi=False`, lalu membandingkannya dengan hasil Kegiatan 1.


### 2.1 Melatih tanpa standarisasi

Sel berikut butuh waktu **2–5 menit**. Sangat mungkin muncul peringatan `ConvergenceWarning` — jangan panik, justru itulah yang ingin kita amati.

In [ ]:
import warnings
from sklearn.exceptions import ConvergenceWarning

print("Melatih TANPA standarisasi (piksel mentah 0-255)...")
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter("always", ConvergenceWarning)
    hasil_tanpa = eksperimen(SEED, gunakan_standarisasi=False)
    ada_peringatan = any(issubclass(x.category, ConvergenceWarning) for x in w)

hasil_dengan = hasil_k1[0]   # hasil dengan standarisasi, SEED yang sama

tabel_k2 = pd.DataFrame([
    {"Perlakuan": "Dengan standarisasi", **hasil_dengan},
    {"Perlakuan": "Tanpa standarisasi", **hasil_tanpa},
])[["Perlakuan", "akurasi_latih", "akurasi_uji", "lama_detik"]]

print("\nMuncul peringatan konvergensi pada versi tanpa standarisasi?", ada_peringatan)
tabel_k2

In [ ]:
# Perbandingan dalam bentuk grafik
plot_df = tabel_k2.melt(id_vars="Perlakuan",
                        value_vars=["akurasi_latih", "akurasi_uji"],
                        var_name="Jenis data", value_name="Akurasi")

px.bar(plot_df, x="Jenis data", y="Akurasi", color="Perlakuan", barmode="group",
       title="Pengaruh standarisasi terhadap akurasi",
       labels={"Jenis data": "", "Akurasi": "Akurasi"},
       height=420)

### 2.2 Tabel Hasil Pengamatan

| Perlakuan | Akurasi latih | Akurasi uji | Lama pelatihan (detik) | Muncul ConvergenceWarning? |
|---|---|---|---|---|
| Dengan standarisasi | ...... | ...... | ...... | ...... |
| Tanpa standarisasi | ...... | ...... | ...... | ...... |

| Ringkasan | Nilai |
|---|---|
| Selisih akurasi uji (dengan − tanpa) | ...... |
| Rasio lama pelatihan (tanpa ÷ dengan) | ...... |


### 2.3 Pertanyaan Analisis

**B1.** Manakah yang lebih terpengaruh oleh standarisasi pada percobaan Anda: **akurasi** atau **lama pelatihan**? Tunjukkan angkanya.

> *Jawaban Anda:*
>
> ......

**B2.** Mengapa nilai piksel mentah 0–255 dapat menyulitkan algoritma optimisasi? Kaitkan jawaban Anda dengan analogi "menuruni bukit" pada materi optimisasi.

> *Jawaban Anda:*
>
> ......

**B3.** Pada dataset ini seluruh fitur memiliki satuan yang sama (intensitas piksel). Bayangkan sebuah dataset tabular berisi kolom `usia` (18–65) dan `pendapatan` (1.000.000–50.000.000). Menurut Anda, apakah dampak standarisasi akan lebih besar atau lebih kecil daripada kasus kita? Jelaskan alasannya.

> *Jawaban Anda:*
>
> ......


---
# Kegiatan 3 — Membandingkan Dua Keluarga Model

**Rujukan:** Latihan mandiri nomor 3 pada `lec03`.

**Pertanyaan yang ingin dijawab:** apakah jaringan saraf mengungguli regresi logistik? Berapa harga yang harus dibayar untuk keunggulan itu?

### Langkah kerja

1. Latih sebuah `MLPClassifier` pada data latih yang sama.
2. Bandingkan akurasi validasi dan lama pelatihannya dengan `lr_model`.
3. Isi tabel dan jawab pertanyaan analisis.

> **Penting.** Perbandingan dilakukan pada **data validasi**, bukan data latih (karena model rumit selalu menang di data latih) dan bukan pula data uji (yang disimpan untuk penilaian akhir).


### 3.1 Melatih jaringan saraf

Lengkapi bagian bertanda `____`. Sel ini butuh waktu **2–5 menit**.

In [ ]:
t0 = time.time()

# TODO (a): buat MLPClassifier dengan dua lapisan tersembunyi berukuran 100 dan 50
mlp = MLPClassifier(
    hidden_layer_sizes=____,
    max_iter=60, tol=1e-3, random_state=SEED)

# TODO (b): latih model pada data latih yang sudah difeaturisasi
mlp.____(X=____, y=labels_tr)

waktu_mlp = time.time() - t0
print("Jaringan saraf selesai dilatih dalam {:.1f} detik.".format(waktu_mlp))

### 3.2 Membandingkan kedua model

In [ ]:
# TODO (c): hitung akurasi VALIDASI untuk kedua model
acc_val_lr = accuracy_score(labels_val, lr_model.predict(____))
acc_val_mlp = accuracy_score(labels_val, ____.predict(X_val))

acc_tr_lr = accuracy_score(labels_tr, lr_model.predict(X_tr))
acc_tr_mlp = accuracy_score(labels_tr, mlp.predict(X_tr))

n_par_lr = lr_model.coef_.size + lr_model.intercept_.size
n_par_mlp = sum(c.size for c in mlp.coefs_) + sum(b.size for b in mlp.intercepts_)

tabel_k3 = pd.DataFrame([
    {"Model": "Regresi Logistik", "Akurasi latih": acc_tr_lr, "Akurasi validasi": acc_val_lr,
     "Lama latih (detik)": waktu_lr, "Jumlah parameter": n_par_lr},
    {"Model": "Jaringan Saraf (MLP)", "Akurasi latih": acc_tr_mlp, "Akurasi validasi": acc_val_mlp,
     "Lama latih (detik)": waktu_mlp, "Jumlah parameter": n_par_mlp},
])

tabel_k3["Selisih latih - validasi"] = tabel_k3["Akurasi latih"] - tabel_k3["Akurasi validasi"]
tabel_k3

### 3.3 Tabel Hasil Pengamatan

| Model | Akurasi latih | Akurasi validasi | Lama latih (detik) | Jumlah parameter | Selisih latih − validasi |
|---|---|---|---|---|---|
| Regresi Logistik | ...... | ...... | ...... | ...... | ...... |
| Jaringan Saraf (MLP) | ...... | ...... | ...... | ...... | ...... |


### 3.4 Pertanyaan Analisis

**C1.** Model mana yang memiliki akurasi validasi lebih tinggi, dan berapa selisihnya?

> *Jawaban Anda:*
>
> ......

**C2.** Sebutkan **tiga harga** yang harus dibayar untuk memakai model yang menang tersebut (lihat kolom lama latih, jumlah parameter, dan selisih latih − validasi).

> *Jawaban Anda:*
>
> ......

**C3.** Kolom "Selisih latih − validasi" adalah indikator overfitting. Model mana yang lebih rawan overfitting berdasarkan percobaan Anda? Jelaskan mengapa hal itu masuk akal ditinjau dari jumlah parameternya.

> *Jawaban Anda:*
>
> ......

**C4.** Situs FashionHub harus memproses 10.000 unggahan foto per hari dengan anggaran server yang terbatas. Model mana yang Anda rekomendasikan? Berikan argumen yang mempertimbangkan akurasi **dan** biaya.

> *Jawaban Anda:*
>
> ......


---
# Kegiatan 4 — Membaca Confusion Matrix

**Rujukan:** Latihan mandiri nomor 4 pada `lec03`.

**Pertanyaan yang ingin dijawab:** kelas mana yang paling sering tertukar, dan apakah kekeliruan itu masuk akal secara visual?

### Langkah kerja

1. Buat confusion matrix pada data validasi.
2. Temukan dua kelas yang paling sering tertukar secara otomatis.
3. Tampilkan contoh gambarnya dan nilai sendiri.


### 4.1 Membuat confusion matrix

Lengkapi bagian bertanda `____`.

In [ ]:
# TODO (a): buat prediksi model regresi logistik pada data VALIDASI
pred_val = lr_model.predict(____)

# TODO (b): hitung confusion matrix (label sebenarnya lebih dulu, baru prediksi)
cm = confusion_matrix(____, pred_val, labels=lr_model.classes_)

fig = px.imshow(cm, color_continuous_scale='Blues', text_auto=True,
                title="Confusion Matrix pada Data Validasi", height=600)
fig.update_layout(
    xaxis_title="Label prediksi",
    yaxis_title="Label sebenarnya",
    coloraxis_showscale=False,
    xaxis=dict(tickmode='array', tickvals=np.arange(len(lr_model.classes_)),
               ticktext=lr_model.classes_),
    yaxis=dict(tickmode='array', tickvals=np.arange(len(lr_model.classes_)),
               ticktext=lr_model.classes_))
fig

### 4.2 Menemukan pasangan kelas yang paling sering tertukar

Sel berikut mencari sel di **luar diagonal** dengan nilai terbesar.

In [ ]:
kelas = lr_model.classes_
cm_luar_diagonal = cm.copy()
np.fill_diagonal(cm_luar_diagonal, 0)

# Ambil 5 kekeliruan terbesar
urut = np.dstack(np.unravel_index(np.argsort(-cm_luar_diagonal, axis=None), cm.shape))[0][:5]

daftar_keliru = pd.DataFrame([
    {"Label sebenarnya": kelas[i], "Diprediksi sebagai": kelas[j], "Jumlah": int(cm[i, j])}
    for i, j in urut
])

print("Lima kekeliruan terbesar model Anda:\n")
daftar_keliru

In [ ]:
# Ambil pasangan kelas yang paling sering tertukar
kelas_asli = daftar_keliru.loc[0, "Label sebenarnya"]
kelas_prediksi = daftar_keliru.loc[0, "Diprediksi sebagai"]

print("Menampilkan gambar berlabel '{}' yang keliru diprediksi sebagai '{}'".format(
    kelas_asli, kelas_prediksi))

mask = (labels_val == kelas_asli) & (pred_val == kelas_prediksi)
idx_keliru = np.where(mask)[0][:5]

show_images(images_val[idx_keliru], max_images=5, ncols=5,
            labels=np.array(["asli: {} / prediksi: {}".format(kelas_asli, kelas_prediksi)] * len(idx_keliru)))

In [ ]:
# Sebagai pembanding: contoh gambar dari kelas yang diprediksi
idx_pembanding = np.where(labels_val == kelas_prediksi)[0][:5]

print("Contoh gambar yang memang berlabel '{}':".format(kelas_prediksi))
show_images(images_val[idx_pembanding], max_images=5, ncols=5,
            labels=np.array([kelas_prediksi] * len(idx_pembanding)))

### 4.3 Tabel Hasil Pengamatan

| Peringkat | Label sebenarnya | Diprediksi sebagai | Jumlah kekeliruan |
|---|---|---|---|
| 1 | ...... | ...... | ...... |
| 2 | ...... | ...... | ...... |
| 3 | ...... | ...... | ...... |


### 4.4 Pertanyaan Analisis

**D1.** Sebutkan dua kelas yang paling sering tertukar pada model Anda, beserta jumlah kekeliruannya.

> *Jawaban Anda:*
>
> ......

**D2.** Perhatikan gambar-gambar yang ditampilkan pada sel 4.2. Sebagai manusia, apakah Anda juga akan keliru membedakan kedua kelas tersebut? Jelaskan ciri visual apa yang membuatnya mirip atau berbeda.

> *Jawaban Anda:*
>
> ......

**D3.** Confusion matrix memberi informasi yang tidak dapat diberikan oleh angka akurasi tunggal. Sebutkan **dua** informasi tersebut.

> *Jawaban Anda:*
>
> ......

**D4.** Berdasarkan temuan Anda, usulkan **satu** perbaikan konkret agar model lebih baik membedakan kedua kelas yang tertukar itu. Usulan boleh menyentuh data, fitur, atau keluarga model.

> *Jawaban Anda:*
>
> ......


---
# Kegiatan 5 — Metrik Selain Akurasi

**Rujukan:** Latihan mandiri nomor 5 pada `lec03`.

**Pertanyaan yang ingin dijawab:** kelas mana yang paling sulit bagi model, dan mengapa akurasi keseluruhan tidak cukup untuk menjawabnya?

### Langkah kerja

1. Hitung *precision*, *recall*, dan *F1-score* untuk setiap kelas.
2. Bandingkan akurasi model dengan baseline tebakan acak.
3. Jawab pertanyaan analisis.


### 5.1 Laporan klasifikasi per kelas

Lengkapi bagian bertanda `____`.

In [ ]:
# TODO (a): buat laporan klasifikasi pada data validasi
#           gunakan classification_report(label_sebenarnya, prediksi, output_dict=True, zero_division=0)
laporan = classification_report(labels_val, ____, output_dict=True, zero_division=0)

tabel_k5 = (pd.DataFrame(laporan).transpose()
              .loc[lr_model.classes_, ["precision", "recall", "f1-score", "support"]]
              .sort_values("f1-score"))

tabel_k5.round(3)

In [ ]:
# Visualisasi F1-score per kelas
px.bar(tabel_k5.reset_index(), x="index", y="f1-score",
       title="F1-Score per Kelas (data validasi)",
       labels={"index": "Kelas", "f1-score": "F1-Score"},
       height=450).update_xaxes(categoryorder="total ascending")

### 5.2 Membandingkan dengan baseline tebakan acak

In [ ]:
np.random.seed(SEED)

akurasi_model = accuracy_score(labels_val, pred_val)
tebakan_acak = np.random.choice(lr_model.classes_, size=len(labels_val))
akurasi_acak = accuracy_score(labels_val, tebakan_acak)

print("Akurasi model        : {:.4f}".format(akurasi_model))
print("Akurasi tebakan acak : {:.4f}".format(akurasi_acak))
print("Model lebih baik {:.1f} kali lipat dibanding menebak acak.".format(
    akurasi_model / akurasi_acak))

### 5.3 Tabel Hasil Pengamatan

*Isi tiga kelas dengan F1-score terendah dan tiga tertinggi dari keluaran sel 5.1.*

| | Kelas | Precision | Recall | F1-Score |
|---|---|---|---|---|
| **Tiga terendah** | ...... | ...... | ...... | ...... |
| | ...... | ...... | ...... | ...... |
| | ...... | ...... | ...... | ...... |
| **Tiga tertinggi** | ...... | ...... | ...... | ...... |
| | ...... | ...... | ...... | ...... |
| | ...... | ...... | ...... | ...... |

| Ringkasan | Nilai |
|---|---|
| Akurasi model (validasi) | ...... |
| Akurasi tebakan acak | ...... |
| Rasio model ÷ acak | ...... |


### 5.4 Pertanyaan Analisis

**E1.** Kelas mana yang paling sulit bagi model Anda (F1-score terendah)? Apakah kelas itu sama dengan kelas yang muncul pada Kegiatan 4?

> *Jawaban Anda:*
>
> ......

**E2.** Pilih satu kelas dengan **precision jauh berbeda dari recall**. Jelaskan dengan bahasa sehari-hari apa arti perbedaan itu bagi pengguna situs FashionHub.

> *Jawaban Anda:*
>
> ......

**E3.** Fashion-MNIST memiliki 10 kelas yang seimbang, sehingga tebakan acak menghasilkan akurasi sekitar 10%. Bagaimana bila datanya timpang — misalnya 90% gambar adalah "Kaos"? Berapa akurasi yang dicapai model yang selalu menjawab "Kaos", dan mengapa akurasi menjadi metrik yang menyesatkan pada kasus seperti itu?

> *Jawaban Anda:*
>
> ......

**E4.** Untuk kasus FashionHub, menurut Anda metrik mana yang paling layak dilaporkan kepada pemilik bisnis: akurasi keseluruhan, F1 rata-rata (macro), atau F1 per kelas? Berikan alasannya.

> *Jawaban Anda:*
>
> ......


---
# Kesimpulan dan Refleksi

### Kesimpulan Praktikum

*Tuliskan minimal lima poin kesimpulan berdasarkan hasil percobaan Anda sendiri, bukan berdasarkan teori umum. Sebutkan angka bila relevan.*

1. ......
2. ......
3. ......
4. ......
5. ......

### Refleksi

**R1.** Bagian mana dari praktikum ini yang paling sulit Anda pahami? Apa yang akhirnya membuat Anda mengerti (atau apa yang masih mengganjal)?

> *Jawaban Anda:*
>
> ......

**R2.** Kesalahan atau error apa yang Anda temui saat mengerjakan, dan bagaimana Anda mengatasinya?

> *Jawaban Anda:*
>
> ......

**R3.** Jika diberi waktu satu minggu lagi untuk memperbaiki model ini, apa **satu** hal pertama yang akan Anda coba? Mengapa itu yang Anda dahulukan?

> *Jawaban Anda:*
>
> ......


---
# Rubrik Penilaian

*Bagian ini diisi oleh dosen atau asisten praktikum.*

| No | Aspek yang Dinilai | Bobot | Skor (0–100) | Nilai |
|---|---|---|---|---|
| 1 | Kelengkapan tabel hasil pengamatan (Kegiatan 1–5) | 20% | | |
| 2 | Ketepatan pengisian kode bertanda `TODO` | 25% | | |
| 3 | Kualitas jawaban analisis (A1–E4) | 35% | | |
| 4 | Kesimpulan dan refleksi | 20% | | |
| | **Nilai Akhir** | **100%** | | |

**Catatan dosen:**

> ......

### Pedoman skor jawaban analisis

| Skor | Kriteria |
|---|---|
| 85–100 | Jawaban tepat, didukung angka dari percobaan sendiri, dan menunjukkan penalaran yang jelas |
| 70–84 | Jawaban tepat tetapi kurang didukung data atau penalarannya dangkal |
| 55–69 | Jawaban sebagian benar atau hanya mengulang teori tanpa mengaitkan hasil percobaan |
| < 55 | Jawaban tidak tepat, kosong, atau merupakan salinan dari mahasiswa lain |


---
# Daftar Periksa Sebelum Mengumpulkan

Centang dengan mengganti `[ ]` menjadi `[x]` (klik dua kali sel ini untuk mengedit).

- [ ] Identitas pada bagian atas sudah diisi lengkap
- [ ] `NIM_3_DIGIT` sudah diganti dengan NIM saya sendiri
- [ ] Semua sel bertanda `TODO` sudah dilengkapi dan berjalan tanpa error
- [ ] Seluruh sel sudah dijalankan berurutan dari atas ke bawah (nomor `In [ ]` berurutan)
- [ ] Semua tabel hasil pengamatan sudah diisi angka dari komputer saya
- [ ] Semua pertanyaan analisis A1–E4 sudah dijawab
- [ ] Kesimpulan minimal lima poin sudah ditulis
- [ ] Refleksi R1–R3 sudah diisi
- [ ] Notebook sudah disimpan (`Ctrl + S`) sebelum diekspor

### Langkah ekspor PDF

1. Simpan notebook: `Ctrl + S`
2. `File` → `Save and Export Notebook As...` → `HTML`
3. Buka berkas HTML hasilnya di browser
4. Tekan `Ctrl + P`, pilih tujuan **Save as PDF**, lalu simpan
5. Beri nama berkas: `LKM03_NIM_NamaLengkap.pdf`

> **Tips.** Pada dialog cetak, aktifkan opsi **Background graphics** agar grafik Plotly ikut tercetak dengan warna. Bila grafik Plotly tetap kosong pada PDF, jalankan sel berikut lebih dahulu lalu ulangi ekspor.


In [ ]:
# Jalankan sel ini bila grafik Plotly tidak muncul saat diekspor ke HTML/PDF
import plotly.io as pio
pio.renderers.default = "notebook"
print("Renderer Plotly diatur ke 'notebook'. Jalankan ulang sel-sel grafik, lalu ekspor kembali.")

---

*Lembar Kerja Mahasiswa — Praktikum Pertemuan 03*
*Materi diadaptasi dari kuliah "Machine Learning Mechanics — Terminology and Techniques" oleh Joseph E. Gonzalez dan Narges Norouzi.*
